# Load packages and cached data
Requires that you ran the extraction step prior and saved the data locally

In [1]:
# Import necessary libraries
from pathlib import Path
import json

import pandas as pd

from fantasy_football.extract.fantasypros import (
    fetch_consensus_adp,
    save_raw_response,
)

In [2]:
# Load data from cache (avoid making repeated API calls)

PROJECT_ROOT = Path.cwd().parent

RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "fantasypros_consensus_adp_2026_half.json"
)

with RAW_PATH.open("r", encoding="utf-8") as f:
    payload = json.load(f)

# Inspect the raw data

In [3]:
# Load json data into a pandas DataFrame
players_df_raw = pd.DataFrame(payload["players"])

print(players_df_raw.shape)

players_df_raw.info()

(340, 25)
<class 'pandas.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   player_id               340 non-null    int64  
 1   player_name             340 non-null    str    
 2   sportsdata_id           340 non-null    str    
 3   player_team_id          340 non-null    str    
 4   player_position_id      340 non-null    str    
 5   player_positions        340 non-null    str    
 6   player_short_name       340 non-null    str    
 7   player_eligibility      340 non-null    str    
 8   player_yahoo_positions  338 non-null    str    
 9   player_page_url         340 non-null    str    
 10  player_filename         340 non-null    str    
 11  player_yahoo_id         340 non-null    str    
 12  cbs_player_id           340 non-null    str    
 13  player_bye_week         305 non-null    str    
 14  player_owned_avg        340 non-null    flo

In [4]:
players_df_raw.head()

,player_id,player_name,sportsdata_id,player_team_id,player_position_id,player_positions,player_short_name,player_eligibility,player_yahoo_positions,player_page_url,...,player_owned_espn,player_owned_yahoo,player_ecr_delta,rank_ecr,rank_min,rank_max,rank_ave,rank_std,pos_rank,tier
0,22968,Jahmyr Gibbs,fef9457e-6497-47de-9bf2-cc3b95929375,DET,RB,RB,J. Gibbs,RB,RB,https://www.fantasypros.com/nfl/players/jahmyr...,...,99.9,100,None,1,1,1,1.00,0.00,RB1,1
1,23133,Bijan Robinson,f78d68c2-f9da-48e7-b954-26b69efd828d,ATL,RB,RB,B. Robinson,RB,RB,https://www.fantasypros.com/nfl/players/bijan-...,...,99.9,100,None,2,2,2,2.00,0.00,RB2,1
2,19788,Ja'Marr Chase,fa99e984-d63b-4ef4-a164-407f68a7eeaf,CIN,WR,WR,J. Chase,WR,WR,https://www.fantasypros.com/nfl/players/jamarr...,...,99.9,100,None,3,3,3,3.00,0.00,WR1,1
3,23180,Puka Nacua,111be44d-7bc2-4cad-934d-c9e946293b2f,LAR,WR,WR,P. Nacua,WR,WR,https://www.fantasypros.com/nfl/players/puka-n...,...,99.9,100,None,4,4,4,4.00,0.00,WR2,1
4,16393,Christian McCaffrey,f96db0af-5e25-42d1-a07a-49b4e065b364,SF,RB,RB,C. McCaffrey,RB,RB,https://www.fantasypros.com/nfl/players/christ...,...,99.9,100,None,5,5,5,5.00,0.00,RB3,2


In [5]:
players_df_raw.isna().sum().sort_values(ascending=False)

player_ecr_delta          340
player_bye_week            35
player_yahoo_positions      2
player_id                   0
player_name                 0
player_position_id          0
player_positions            0
player_team_id              0
sportsdata_id               0
player_eligibility          0
player_short_name           0
player_yahoo_id             0
player_page_url             0
cbs_player_id               0
player_owned_avg            0
player_owned_espn           0
player_filename             0
player_owned_yahoo          0
rank_ecr                    0
rank_min                    0
rank_max                    0
rank_ave                    0
rank_std                    0
pos_rank                    0
tier                        0
dtype: int64

## Trim & organize the raw data

In [ ]:
organized_column_list = [
    
    'player_name', 
    'player_team_id',
    'player_position_id', 
    'player_eligibility', # Not same as player_position_id, but unknown how --> Keep for now
    'pos_rank', 
    'player_bye_week', # move this down? 
    
    'tier', # TBD - overall or position? Seems to be overall 
    'rank_ecr', 
    'rank_min',
    'rank_max', 
    'rank_ave', 
    'rank_std', 

    # save for when cross-platform ADP is figured out:
    'player_id', 'sportsdata_id','player_yahoo_id', 'cbs_player_id', 

    # ### Dropped Fields:
    # 'player_yahoo_positions', # Not same as player_position_id, but not drafting on Yahoo --> Drop for now
    # 'player_positions', # Same as player_position_id as of 8/27/26
    # 'player_short_name', # Redundant to name
    # 'player_page_url', # Not needed
    # 'player_filename', # Not needed    
    # 'player_owned_avg', 'player_owned_espn', 'player_owned_yahoo', # Not that insightful
    # 'player_ecr_delta', # 100% null
    
    ]

players_df = players_df_raw[organized_column_list].copy()

In [11]:
# Rename columns
rename_dict = {
    "player_team_id": "team",
    "player_position_id": "position",
    "player_eligibility": "eligibile_positions",
    "player_bye_week": "bye_wk",
    "rank_ave": "rank_avg"
}

players_df = players_df.rename(columns=rename_dict)

# Enrich the cleaned player data

In [8]:
players_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   player_name          340 non-null    str  
 1   team                 340 non-null    str  
 2   position             340 non-null    str  
 3   eligibile_positions  340 non-null    str  
 4   bye_wk               305 non-null    str  
 5   pos_rank             340 non-null    str  
 6   tier                 340 non-null    int64
 7   rank_ecr             340 non-null    int64
 8   rank_min             340 non-null    str  
 9   rank_max             340 non-null    str  
 10  rank_avg             340 non-null    str  
 11  rank_std             340 non-null    str  
 12  player_id            340 non-null    int64
 13  sportsdata_id        340 non-null    str  
 14  player_yahoo_id      340 non-null    str  
 15  cbs_player_id        340 non-null    str  
dtypes: int64(3), str(13)
memory usage: 42

In [9]:
players_df.head(5)

,player_name,team,position,eligibile_positions,bye_wk,pos_rank,tier,rank_ecr,rank_min,rank_max,rank_avg,rank_std,player_id,sportsdata_id,player_yahoo_id,cbs_player_id
0,Jahmyr Gibbs,DET,RB,RB,6,RB1,1,1,1,1,1.00,0.00,22968,fef9457e-6497-47de-9bf2-cc3b95929375,40059,3162723
1,Bijan Robinson,ATL,RB,RB,11,RB2,1,2,2,2,2.00,0.00,23133,f78d68c2-f9da-48e7-b954-26b69efd828d,40055,3168163
2,Ja'Marr Chase,CIN,WR,WR,6,WR1,1,3,3,3,3.00,0.00,19788,fa99e984-d63b-4ef4-a164-407f68a7eeaf,33393,2966320
3,Puka Nacua,LAR,WR,WR,11,WR2,1,4,4,4,4.00,0.00,23180,111be44d-7bc2-4cad-934d-c9e946293b2f,40168,3121687
4,Christian McCaffrey,SF,RB,RB,8,RB3,2,5,5,5,5.00,0.00,16393,f96db0af-5e25-42d1-a07a-49b4e065b364,30121,2136743
